# Boarding NoteBook - NeuralHydrology - FK

## Objectives
- From scratch, read the doc, understand how the pieces are build and then run together
- links: 
  - [Doc](https://neuralhydrology.readthedocs.io/en/latest/index.html) 
  - [GitHub](https://github.com/neuralhydrology) 
  - [Issue of the task for FK](https://github.com/hydrique/ml-lab/issues/10) 


## Day Zero - 29.09.2025
Only started in the afternoon, for the moment beginning of the tutorial.

Questions: 
- Dataset for tuto is very heavy, i will try to go through the tuto without running the code locally, such that i wont need to DL the dataset (40 Gb)

Done:
- Folder with my first tests for now: C:\Users\Kohler\Documents\NeuralHydrology_FK
- There is also the cloned repo of NH

- Creating virtual environment, location here: "C:\Users\Kohler\Documents\NeuralHydrology_FK\.venv"

- Tuto DL Dataset, for now just read, not downloaded - its too big (https://neuralhydrology.readthedocs.io/en/latest/tutorials/data-prerequisites.html)
- 


### Notes - Tuto
**Ways to use NH:**
1. From terminal (cmd such as `nh-run train` to train or `nh-schedule-runs train..` to train multiple models)
   - `nh-run train --config-file path/to/config.yml`
   - `nh-run evaluate --run-dir path/to/run-directory`
   - Multiple models on different GPUs - automatic distribution between gpus
   - `nh-schedule-runs train --directory /path/to/config-dir --runs-per-gpu 2 --gpu_ids 0 1 2 3` (GPU id according to Nvidia-smi)
   - `nh-schedule-runs evaluate --directory /path/to/parent-run-dir/ --runs-per-gpu 2 --gpu_ids 0 1 2 3`
   - 
2. From a python or jupyterNB with NH's API
   - [Doc](https://neuralhydrology.readthedocs.io/en/latest/api/neuralhydrology.html)

**Requirements for a Run**
1. Config file (`.yml` files)
   - defines run configuration - DS, bassins, time periods, model,...
   - complete list of arguments [HERE](https://neuralhydrology.readthedocs.io/en/latest/usage/config.html) 
2. New run = new folder, will stock:
   - model & optimizer checkpoints, training Data statistics (mean, stds), Tensorboard log file (monitoring, visual comparison of runs)
3. **cuda - cpu**:
   - either change the config argument to device: cpu or pass gpu=-1 to the start_run() function
4. s

### New model
How to add a new model in the NH Modelzoo.
- Template [HERE](https://github.com/neuralhydrology/neuralhydrology/blob/master/neuralhydrology/modelzoo/template.py)
- NOTE common base is BaseModel (in modelzoo)
- constructor has 1 argument: an instance of the **Config class**
- proper logic in the forward method (from input to prediction=output)
- A model is in 2 parts
  1. Head
        - relates the model class output to the predicted variabele 
        - 4 heads currently implemented: Regression, GMM CMAL, UMAL (last 3 for proba modelling)
        - paper to read about it [here](https://arxiv.org/abs/2012.14295)
  2. Class

#### example: build a new type of GRU Model
- Once you have implemented your model, make sure to modify neuralhydrology.modelzoo.__init__.get_model(). Furthermore, make sure to select a unique model abbreviation that will be used to specify the model in the config.yml files.
- Most of the models have three components:
    1. Input layer (optionnal): embdedding network for both static or dynamic features
    2. Body 
    3. Head = final output layer
- For modular architecture, head and input not in the new model, use InputLayer and get_head fct. Construction of the layer automatic such that it fits the config parameters

**Forward method**
- takes and gives dictionnaries - mapping string to tensors (data countains x_d and sometimes x_s, x_one_hot)
- output `y_hat` or `y_hat_xh` for multiscale simulation
- The naming convention for hidden states is to call them ‘h_n’.
- The input layer merges the static inputs (data['x_s'] and/or data['x_one_hot']) to each step of the dynamic inputs (data['x_d']) and returns a single tensor that we can pass to the GRU cell.


- The only thing left is registering the model in the get_model method of neuralhydrology.modelzoo to make sure we can specify the model in a run configuration.

### Build new dataset
2 diff options:
1. Reprocessing of the data in order to use the class `GenericDataset`
2. Keep formatting of the data as now and implement new DS class
    should be this option from my pov, depending on the format necessary for the class already implemented

**DATA FORMATTING - for GenericDataset**
- directory (data_dir in config) must have `time_series` folder and a `attributes` folder to store static attributes
- Folder `time_series` must contain one *netcdf* file (.nc or .nc4) per basin - named "*basin_id*.nc". Must contain 1 coordinate `date` with datetime index
- Folder `attributes` - one or more csv with static attributes, indexed by basin_id. Attributes file separated into groups of basins or groups of features.
- NOTE invalid data must be named as NaN 

**BaseDataset**
- has classical Pytorch DS methods and attributes:
  - `__len__` = number of total training samples
  - `__getitem__` single training sample for a given index


**DS class**
Template for DS classes [here](https://github.com/neuralhydrology/neuralhydrology/blob/master/neuralhydrology/datasetzoo/template.py)
Important points: 
- inherit from BaseDataset
- all classes should accept same input upon init.
- for each class, implement these two methods
    1. `_load_basin_data()` load TS for a single basin into a time-indexed pd.DataFrame (panda)
    2. `_load_attributes()_` load catchment attributes for all basins and returns a basin-indexed pd.DataFrame with attributes as columns
- Data loading functions (extract from csv or txt)
  - time series
  - attributes

example:

In [4]:
class CamelsCL(BaseDataset):

    def __init__(self,
                 cfg: Config,
                 is_train: bool,
                 period: str,
                 basin: str = None,
                 additional_features: List[Dict[str, pd.DataFrame]] = [],
                 id_to_int: Dict[str, int] = {},
                 scaler: Dict[str, Union[pd.Series, xarray.DataArray]] = {}):

        # Initialize `BaseDataset` class
        super(CamelsCL, self).__init__(cfg=cfg,
                                       is_train=is_train,
                                       period=period,
                                       basin=basin,
                                       additional_features=additional_features,
                                       id_to_int=id_to_int,
                                       scaler=scaler)

    def _load_basin_data(self, basin: str) -> pd.DataFrame:
        """Load timeseries data of one specific basin"""
        return load_camels_cl_timeseries(data_dir=self.cfg.data_dir, basin=basin)

    def _load_attributes(self) -> pd.DataFrame:
        """Load catchment attributes"""
        return load_camels_cl_attributes(self.cfg.data_dir, basins=self.basins)

NameError: name 'BaseDataset' is not defined

The next step is to implement the new class in the NH framework:
1. Static way - in the __init__.py file

   _datasetZooRegistry.register_dataset_class("camels_cl", CamelsCL) # Just add this line to register 'CamelsC'L' class
2. Dynamic way - to use NH as a library and dont want to change lines of code -> use `register_dataset()`

**** 
## Example of a complete run with the dataset for training just to see the steps

Set the config file precisely and then it becomes pretty easy to do the complete training and evaluation

In [1]:
import pickle
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from neuralhydrology.evaluation import metrics
from neuralhydrology.nh_run import start_run, eval_run

C:\Users\Kohler\Documents\NeuralHydrology_FK\neuralhydrology\neuralhydrology\datautils\utils.py:242: SyntaxWarning: invalid escape sequence '\d'
  weekly_freq = re.match('(\d+)W(-(MON|TUE|WED|THU|FRI|SAT|SUN))?$', native_frequency)
C:\Users\Kohler\Documents\NeuralHydrology_FK\neuralhydrology\neuralhydrology\datasetzoo\camelsus.py:184: SyntaxWarning: invalid escape sequence '\s'
  df = pd.read_csv(fp, sep='\s+')
C:\Users\Kohler\Documents\NeuralHydrology_FK\neuralhydrology\neuralhydrology\datasetzoo\camelsus.py:220: SyntaxWarning: invalid escape sequence '\s'
  df = pd.read_csv(file_path, sep='\s+', header=None, names=col_names)


In [2]:
# by default we assume that you have at least one CUDA-capable NVIDIA GPU or MacOS with Metal support
if torch.cuda.is_available() or torch.backends.mps.is_available():
    start_run(config_file=Path("1_basin.yml"))

# fall back to CPU-only mode
else:
    start_run(config_file=Path("1_basin.yml"), gpu=-1)

FileNotFoundError: 1_basin.yml

Evaluate on Test set

In [ ]:
run_dir = Path("runs/test_run_0501_214945")
eval_run(run_dir=run_dir, period="test")

Load to inspect predictions

In [ ]:
with open(run_dir / "test" / "model_epoch050" / "test_results.p", "rb") as fp:
    results = pickle.load(fp)

results.keys()

Plot model predictions vs. observations

In [ ]:
# extract observations and simulations
qobs = results['01022500']['1D']['xr']['QObs(mm/d)_obs']
qsim = results['01022500']['1D']['xr']['QObs(mm/d)_sim']

fig, ax = plt.subplots(figsize=(16,10))
ax.plot(qobs['date'], qobs)
ax.plot(qsim['date'], qsim)
ax.set_ylabel("Discharge (mm/d)")
ax.set_title(f"Test period - NSE {results['01022500']['1D']['NSE']:.3f}")

Evaluate the clasical metrics

In [1]:
values = metrics.calculate_all_metrics(qobs.isel(time_step=-1), qsim.isel(time_step=-1))
for key, val in values.items():
    print(f"{key}: {val:.3f}")

NameError: name 'metrics' is not defined

****

## MTS-prediction 
MTS = Multi-Timescale prediction
Meteorological forcing variables = key climate factors, that drive environmental models
- $T_{min}$, $T_{max}$, Qp (precipitation), solar radiation, wind speed, pressure, humidity, etc.

Better to train on multiple basins **Find some papers - to know the magnitude of the gain in the score of the prediction** - have some numbers
- In hydrological modelling, each basin provides time series data (streamflow/discharge) plus forcings (rainfall, temperature, etc.), and static attributes (topography, land cover, soils, etc.).

- From what i learned, it is non necessary to use multiple bassins when we want to train a model pro App. If we would have wanted a single model for all our system, it wouold have be meaningfull tho.

**Question** 
Would it be worth to train on an huge Dataset, Eu-central, CH (pretraining) and then fine-tune this model to each of our app ?
At which scale would it be advantageous ? train on all our app and then fine tune pro app ?


Notes on MTS-LSTM:
if we have hourly and daily forcing variables and want to have also hourly and daily predictions, train two models (one pro time scale of the prediction) is too heavy for each predictions, takes too much time to train, is simply not efficient
No correlations between hourly and daily pred if there is two indep models, could have two completely pred for the 1st hour of the daily...

instead: single model for both time scale of prediction
**Idea** adapt the time resolution of the input with past time. Measures from 4 months taken daily instead of hourly, 3 weeks before are still taken hourly, in this kind of idea

**Implementation**
lala


****